In [1]:
# ============================================================
# MONTAR DRIVE + VERIFICAR GPU (ejecutar primero)
# ============================================================
import torch
from google.colab import drive

# Montar Drive
drive.mount('/content/drive', force_remount=False)

# Verificar GPU
device = 'CUDA' if torch.cuda.is_available() else 'CPU'
print(f"⚡ Dispositivo: {device}")
if device == 'CPU':
    raise RuntimeError("❌ GPU no disponible — cambia runtime a T4 GPU")
else:
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from pathlib import Path
BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
print(f"\nBASE: {BASE}")
print(f"  Existe: {BASE.exists()}")


Mounted at /content/drive
⚡ Dispositivo: CUDA
  GPU: Tesla T4
  VRAM: 15.6 GB

BASE: /content/drive/MyDrive/TFM/CAIDAS
  Existe: True


# 🔗 Unificación de Datasets: COCO_2017 + YOLO_DATASET_UNIFIED
## TFM — Detección de Caídas

**Estructura real verificada en Google Drive:**

```
My Drive/TFM/CAIDAS/
├── COCO_2017/
│   └── poses/
│       ├── filtrar_poses.py
│       ├── annotations/         ← JSONs COCO originales
│       └── dataset/             ← Dataset YOLO convertido
│           ├── poses.yaml       ← clases del dataset COCO
│           ├── train/
│           │   ├── images/      ← imágenes de entrenamiento
│           │   └── labels/      ← labels YOLO .txt
│           └── val/
│               ├── images/
│               └── labels/
└── YOLO_DATASET_UNIFIED/        ← DESTINO (estructura plana)
    ├── images/                  ← todas las imágenes
    └── labels/                  ← todos los labels .txt
```

**Objetivo:** Copiar todas las imágenes y labels de `COCO_2017/poses/dataset/{train,val}`  
hacia `YOLO_DATASET_UNIFIED/{images,labels}`, remapeando IDs de clases según `poses.yaml`.

> ⚡ **Entorno:** GPU T4 — *Runtime → Change runtime type → T4 GPU*

---
**Celdas:**
1. Verificar GPU y montar Google Drive  
2. Inspeccionar estructura de carpetas  
3. Leer y comparar clases (`poses.yaml` ↔ `YOLO_DATASET_UNIFIED`)  
4. Unificación: copiar COCO→YOLO con remapeo de IDs  
5. Actualizar/crear `data.yaml` con clases unificadas  
6. Verificación final del dataset unificado

In [ ]:
# ============================================================
# DIAGNÓSTICO RÁPIDO — estado actual de YOLO_DATASET_UNIFIED
# Solo usa os.listdir (sin conteo de archivos grandes)
# ============================================================
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
YOLO = BASE / "YOLO_DATASET_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def quick_count(p):
    """Cuenta archivos en UN nivel, sin recursión."""
    try:
        entries = os.listdir(p)
        imgs = sum(1 for e in entries if Path(e).suffix.lower() in IMG_EXTS and not e.startswith("._"))
        txts = sum(1 for e in entries if e.endswith(".txt") and not e.startswith("._"))
        dirs = [e for e in entries if (p / e).is_dir()]
        return imgs, txts, dirs
    except:
        return 0, 0, []

print("=" * 60)
print("YOLO_DATASET_UNIFIED/images/")
imgs_root, _, subdirs_img = quick_count(YOLO / "images")
print(f"  Archivos en RAÍZ    : {imgs_root:,} imágenes")
print(f"  Subcarpetas         : {subdirs_img}")
for sd in subdirs_img:
    n, _, _ = quick_count(YOLO / "images" / sd)
    print(f"    └── {sd}/  {n:,} imágenes")

print()
print("YOLO_DATASET_UNIFIED/labels/")
_, lbls_root, subdirs_lbl = quick_count(YOLO / "labels")
print(f"  Archivos en RAÍZ    : {lbls_root:,} labels .txt")
print(f"  Subcarpetas         : {subdirs_lbl}")
for sd in subdirs_lbl:
    _, n, _ = quick_count(YOLO / "labels" / sd)
    print(f"    └── {sd}/  {n:,} labels")

print()
print("=" * 60)
print("CONCLUSIÓN:")
total_root = imgs_root + lbls_root
if imgs_root == 0 and lbls_root == 0:
    print("✅ No hay archivos en raíz. La estructura ya está correcta.")
    print("   → Ejecuta directamente Celda 6 y Celda 7.")
elif subdirs_img or subdirs_lbl:
    print(f"⚠️  HAY {imgs_root:,} imágenes y {lbls_root:,} labels en la raíz")
    print(f"   Y también existen subcarpetas {subdirs_img}.")
    print("   → Necesitas la Celda 4-A para reorganizar.")
else:
    print(f"ℹ️  Solo hay raíz (sin subcarpetas) — estructura plana.")
    print("   → Ejecuta directamente Celda 6 y Celda 7.")

YOLO_DATASET_UNIFIED/images/
  Archivos en RAÍZ    : 0 imágenes
  Subcarpetas         : ['val', 'train']
    └── val/  16,334 imágenes
    └── train/  0 imágenes

YOLO_DATASET_UNIFIED/labels/
  Archivos en RAÍZ    : 0 labels .txt
  Subcarpetas         : ['train', 'val']
    └── train/  0 labels
    └── val/  16,334 labels

CONCLUSIÓN:
✅ No hay archivos en raíz. La estructura ya está correcta.
   → Ejecuta directamente Celda 6 y Celda 7.


In [ ]:
# ============================================================
# Leer y comparar clases de ambos datasets
# ============================================================
# Archivos de clases verificados:
#   COCO origen : COCO_2017/poses/dataset/poses.yaml
#   YOLO destino: YOLO_DATASET_UNIFIED/data.yaml  (si existe)
#                 o YOLO_DATASET_UNIFIED/classes.txt
# ============================================================
import yaml, os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
SRC  = BASE / "COCO_2017" / "poses" / "dataset"
YOLO = BASE / "YOLO_DATASET_UNIFIED"

# --- Leer clases de COCO (poses.yaml) ---
poses_yaml = SRC / "poses.yaml"
coco_classes = {}
if poses_yaml.exists():
    with open(poses_yaml) as f:
        d = yaml.safe_load(f)
    names = d.get("names", {})
    coco_classes = {i: n for i, n in enumerate(names)} if isinstance(names, list) else names
    print(f"✅ COCO (poses.yaml):")
    print(f"   nc     = {d.get('nc', len(coco_classes))}")
    print(f"   names  = {coco_classes}")
    print(f"   train  = {d.get('train', '—')}")
    print(f"   val    = {d.get('val', '—')}")
else:
    print(f"❌ No se encontró poses.yaml en {SRC}")

print()

# --- Leer clases de YOLO_DATASET_UNIFIED ---
yolo_classes = {}
yolo_yaml_path = None
for fname in ["data.yaml", "dataset.yaml"]:
    yf = YOLO / fname
    if yf.exists():
        with open(yf) as f:
            d = yaml.safe_load(f)
        names = d.get("names", {})
        yolo_classes = {i: n for i, n in enumerate(names)} if isinstance(names, list) else names
        yolo_yaml_path = yf
        print(f"✅ YOLO ({fname}):")
        print(f"   nc    = {d.get('nc', len(yolo_classes))}")
        print(f"   names = {yolo_classes}")
        break
cf = YOLO / "classes.txt"
if not yolo_classes and cf.exists():
    with open(cf) as f:
        lst = [l.strip() for l in f if l.strip()]
    yolo_classes = {i: n for i, n in enumerate(lst)}
    yolo_yaml_path = cf
    print(f"✅ YOLO (classes.txt): {yolo_classes}")

if not yolo_classes:
    print(f"ℹ️  YOLO_DATASET_UNIFIED no tiene archivo de clases todavía (se creará en Celda 5).")

print()
print("=" * 60)
print("📊 Análisis comparativo:")
print(f"   COCO  poses.yaml  : {len(coco_classes)} clases")
print(f"   YOLO  unified     : {len(yolo_classes)} clases")

if coco_classes and yolo_classes:
    common = set(coco_classes.values()) & set(yolo_classes.values())
    new_c  = set(coco_classes.values()) - set(yolo_classes.values())
    print(f"\n   ✅ Comunes  : {sorted(common) if common else 'Ninguna'}")
    print(f"   ➕ Nuevas   : {sorted(new_c) if new_c else 'Ninguna (todas ya existen)'}")

✅ COCO (poses.yaml):
   nc     = 1
   names  = {0: 'person'}
   train  = train/images
   val    = val/images

ℹ️  YOLO_DATASET_UNIFIED no tiene archivo de clases todavía (se creará en Celda 5).

📊 Análisis comparativo:
   COCO  poses.yaml  : 1 clases
   YOLO  unified     : 0 clases


In [ ]:
# ============================================================
# REORGANIZACIÓN RÁPIDA con os.rename
# Mueve archivos de la raíz de images/ y labels/ a subcarpetas
# IMPORTANTE: os.rename NO copia bytes — es un rename de path
#              en Google Drive Colab (operación casi instantánea)
# ============================================================
# ESTADO ACTUAL VERIFICADO:
#   images/  → 11,996 archivos coco_* en RAÍZ + subcarpetas val/ y train/
#   labels/  → misma situación
#
# ACCIÓN: mover coco_train_* → images/train/  labels/train/
#                coco_val_*   → images/val/    labels/val/
# ============================================================
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
YOLO = BASE / "YOLO_DATASET_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DST_IMGS = YOLO / "images"
DST_LBLS = YOLO / "labels"

# Crear subcarpetas si faltan
for split in ["train", "val"]:
    (DST_IMGS / split).mkdir(parents=True, exist_ok=True)
    (DST_LBLS / split).mkdir(parents=True, exist_ok=True)

PREFIX_MAP = {
    "coco_train_" : "train",
    "coco_val_"   : "val",
    "coco_test_"  : "test",
}

moved_imgs = 0
moved_lbls = 0
skipped    = 0

print("🔄 Reorganizando imágenes de la raíz...")
print("   (usando os.rename — sin copia de bytes, muy rápido)\n")

# ── Imágenes en raíz ─────────────────────────────────────────
root_imgs = [p for p in DST_IMGS.iterdir()
             if p.is_file() and p.suffix.lower() in IMG_EXTS
             and not p.name.startswith("._")]

print(f"   {len(root_imgs):,} imágenes en raíz de images/")
for img in root_imgs:
    split = next((s for pfx, s in PREFIX_MAP.items() if img.stem.startswith(pfx)), None)
    if split is None:
        split = "train"   # originales sin prefijo van a train
    dst = DST_IMGS / split / img.name
    if dst.exists():
        img.unlink()      # ya existe en destino, eliminar duplicado
        skipped += 1
    else:
        os.rename(img, dst)
        moved_imgs += 1

print(f"   ✅ {moved_imgs:,} imágenes movidas | {skipped} ya existían")

# ── Labels en raíz ───────────────────────────────────────────
root_lbls = [p for p in DST_LBLS.iterdir()
             if p.is_file() and p.suffix == ".txt"
             and not p.name.startswith("._")]

print(f"\n   {len(root_lbls):,} labels en raíz de labels/")
skipped = 0
for lbl in root_lbls:
    split = next((s for pfx, s in PREFIX_MAP.items() if lbl.stem.startswith(pfx)), None)
    if split is None:
        split = "train"
    dst = DST_LBLS / split / lbl.name
    if dst.exists():
        lbl.unlink()
        skipped += 1
    else:
        os.rename(lbl, dst)
        moved_lbls += 1

print(f"   ✅ {moved_lbls:,} labels movidos | {skipped} ya existían")

# ── Resumen final rápido (solo os.listdir) ───────────────────
print()
print("=" * 55)
print("📊 Estado post-reorganización:")
for split in ["train", "val"]:
    sd_img = DST_IMGS / split
    sd_lbl = DST_LBLS / split
    try:
        n_imgs = sum(1 for e in os.listdir(sd_img) if Path(e).suffix.lower() in IMG_EXTS)
        n_lbls = sum(1 for e in os.listdir(sd_lbl) if e.endswith(".txt"))
    except:
        n_imgs, n_lbls = 0, 0
    print(f"   [{split:5s}] images: {n_imgs:>7,}  |  labels: {n_lbls:>7,}")

# Verificar si quedaron archivos en raíz
remaining_imgs = sum(1 for p in DST_IMGS.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)
remaining_lbls = sum(1 for p in DST_LBLS.iterdir() if p.is_file() and p.suffix == ".txt")
print(f"\n   Raíz images/: {remaining_imgs} archivos restantes")
print(f"   Raíz labels/: {remaining_lbls} archivos restantes")
if remaining_imgs == 0 and remaining_lbls == 0:
    print("   ✅ Raíz limpia — estructura correcta")
else:
    print("   ⚠️  Aún quedan archivos en raíz")

🔄 Reorganizando imágenes de la raíz...
   (usando os.rename — sin copia de bytes, muy rápido)

   11,996 imágenes en raíz de images/
   ✅ 11,996 imágenes movidas | 0 ya existían

   11,994 labels en raíz de labels/
   ✅ 11,994 labels movidos | 0 ya existían

📊 Estado post-reorganización:
   [train] images:  87,130  |  labels:  87,128
   [val  ] images:  30,268  |  labels:  30,268

   Raíz images/: 0 archivos restantes
   Raíz labels/: 0 archivos restantes
   ✅ Raíz limpia — estructura correcta


In [ ]:
# ============================================================
# Unificación COCO_2017 → YOLO_DATASET_UNIFIED
#
# ORIGEN  : COCO_2017/poses/dataset/{train,val}/{images,labels}
# DESTINO : YOLO_DATASET_UNIFIED/{images,labels}/{train,val}/
#            (respeta la estructura con subcarpetas existente)
#
# Proceso:
#   1. Detectar si destino usa subcarpetas (train/val) o estructura plana
#   2. Leer clases de poses.yaml y data.yaml/classes.txt
#   3. Construir mapeo de IDs: COCO-ID → YOLO-ID (por nombre)
#   4. Copiar imagen con prefijo "coco_<split>_<stem>" sin colisiones
#   5. Reescribir labels remapeando class_id en cada línea
# ============================================================
import os, shutil, yaml, pickle
from pathlib import Path
from tqdm.auto import tqdm
import torch

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
SRC  = BASE / "COCO_2017" / "poses" / "dataset"
YOLO = BASE / "YOLO_DATASET_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS   = ["train", "val", "test"]

# --- Confirmar GPU ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Dispositivo: {device.upper()}")
if device != "cuda":
    raise RuntimeError("GPU no disponible. Ve a Runtime > Change runtime type > T4 GPU")

# ── Detectar estructura destino ──────────────────────────────
# Si images/train/ existe → usar subcarpetas; si no → estructura plana
USE_SPLITS = (YOLO / "images" / "train").exists() or (YOLO / "images" / "val").exists()
print(f"\n📂 Estructura destino: {'con subcarpetas (train/val)' if USE_SPLITS else 'plana (raíz)'}")

def get_dst_dirs(split):
    """Retorna (img_dir_destino, lbl_dir_destino) según estructura."""
    if USE_SPLITS:
        img_d = YOLO / "images" / split
        lbl_d = YOLO / "labels" / split
    else:
        img_d = YOLO / "images"
        lbl_d = YOLO / "labels"
    img_d.mkdir(parents=True, exist_ok=True)
    lbl_d.mkdir(parents=True, exist_ok=True)
    return img_d, lbl_d

# ── Leer clases de COCO (poses.yaml) ────────────────────────
poses_yaml = SRC / "poses.yaml"
if not poses_yaml.exists():
    raise FileNotFoundError(f"No se encontró poses.yaml en {SRC}")
with open(poses_yaml) as f:
    poses_data = yaml.safe_load(f)
coco_names   = poses_data.get("names", [])
coco_classes = {i: n for i, n in enumerate(coco_names)} if isinstance(coco_names, list) else coco_names
print(f"\n📋 Clases COCO (poses.yaml): {coco_classes}")

# ── Leer clases de YOLO_DATASET_UNIFIED ─────────────────────
yolo_classes   = {}
yolo_yaml_path = None
yolo_yaml_data = {}
for fname in ["data.yaml", "dataset.yaml"]:
    yf = YOLO / fname
    if yf.exists():
        with open(yf) as f:
            yolo_yaml_data = yaml.safe_load(f) or {}
        names = yolo_yaml_data.get("names", {})
        yolo_classes = {i: n for i, n in enumerate(names)} if isinstance(names, list) else names
        yolo_yaml_path = yf
        break
cf = YOLO / "classes.txt"
if not yolo_classes and cf.exists():
    with open(cf) as f:
        lst = [l.strip() for l in f if l.strip()]
    yolo_classes = {i: n for i, n in enumerate(lst)}
    yolo_yaml_path = cf
print(f"📋 Clases YOLO destino: {yolo_classes}")

# ── Mapeo ID COCO → ID YOLO ──────────────────────────────────
yolo_name_to_id  = {v: k for k, v in yolo_classes.items()}
extended_classes = dict(yolo_classes)
next_id = max(yolo_classes.keys(), default=-1) + 1
coco_id_to_yolo_id = {}
for cid, cname in coco_classes.items():
    if cname in yolo_name_to_id:
        coco_id_to_yolo_id[cid] = yolo_name_to_id[cname]
    else:
        extended_classes[next_id] = cname
        coco_id_to_yolo_id[cid]   = next_id
        print(f"  ➕ Nueva clase: '{cname}' → ID {next_id}")
        next_id += 1
print(f"\n🗺️  Mapeo COCO-ID → YOLO-ID: {coco_id_to_yolo_id}")

# ── Procesar cada split ──────────────────────────────────────
stats = {"copiadas": 0, "omitidas": 0, "labels_ok": 0, "labels_vacios": 0}

for split in SPLITS:
    src_img = SRC / split / "images"
    src_lbl = SRC / split / "labels"
    if not src_img.exists():
        print(f"\n⏭️  Split '{split}' no encontrado en origen, se omite.")
        continue

    dst_img, dst_lbl = get_dst_dirs(split)

    # Stems ya existentes en destino (evitar duplicados)
    existing_stems = {p.stem for p in dst_img.iterdir() if p.suffix.lower() in IMG_EXTS}
    images = sorted([p for p in src_img.iterdir() if p.suffix.lower() in IMG_EXTS])
    print(f"\n📂 [{split.upper()}] {len(images):,} imágenes origen | {len(existing_stems):,} ya en destino")
    print(f"   → destino: {dst_img.relative_to(BASE)}")

    for img_path in tqdm(images, desc=f"  [{split}] copiando"):
        # Saltar metadatos macOS
        if img_path.name.startswith("._"):
            continue
        new_stem = f"coco_{split}_{img_path.stem}"
        if new_stem in existing_stems:
            stats["omitidas"] += 1
            continue

        # Copiar imagen
        shutil.copy2(img_path, dst_img / (new_stem + img_path.suffix))
        stats["copiadas"] += 1
        existing_stems.add(new_stem)

        # Procesar label
        src_lbl_file = src_lbl / (img_path.stem + ".txt") if src_lbl.exists() else None
        dst_lbl_file = dst_lbl / (new_stem + ".txt")

        if src_lbl_file and src_lbl_file.exists() and not src_lbl_file.name.startswith("._"):
            new_lines = []
            try:
                with open(src_lbl_file, encoding='utf-8', errors='replace') as f:
                    for line in f:
                        parts = line.strip().split()
                        if not parts or not parts[0].lstrip('-').isdigit():
                            continue
                        old_id = int(parts[0])
                        new_id = coco_id_to_yolo_id.get(old_id, old_id)
                        if len(parts) >= 5:
                            try: _ = [float(x) for x in parts[1:5]]
                            except ValueError: continue
                        new_lines.append(f"{new_id} " + " ".join(parts[1:]))
            except Exception:
                new_lines = []
            with open(dst_lbl_file, "w") as f:
                f.write("\n".join(new_lines))
            stats["labels_ok"] += 1
        else:
            dst_lbl_file.touch()
            stats["labels_vacios"] += 1

print(f"\n{'=' * 60}")
print("✅ Unificación completada")
print(f"   📸 Imágenes copiadas     : {stats['copiadas']:>8,}")
print(f"   ⏭️  Omitidas (duplicadas) : {stats['omitidas']:>8,}")
print(f"   🏷️  Labels copiados       : {stats['labels_ok']:>8,}")
print(f"   ⚠️  Labels vacíos         : {stats['labels_vacios']:>8,}")

with open("/tmp/extended_classes.pkl", "wb") as f:
    import pickle
    pickle.dump({
        "extended_classes"   : extended_classes,
        "yolo_yaml_path"     : str(yolo_yaml_path) if yolo_yaml_path else None,
        "yolo_yaml_data"     : yolo_yaml_data,
        "coco_id_to_yolo_id" : coco_id_to_yolo_id,
        "use_splits"         : USE_SPLITS,
    }, f)
print("\n💾 Datos guardados en /tmp/extended_classes.pkl")

✅ Clases unificadas (1): ['person']

✅ data.yaml guardado: /content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED/data.yaml
   nc    : 1
   names : ['person']
   train : /content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED/images
   val   : /content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED/images


In [13]:
# ============================================================
# Conteo de archivos en carpetas específicas
# ============================================================
import time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
YOLO = BASE / "YOLO_DATASET_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_files_in_dir(directory_path, file_type="any"):
    """Cuenta archivos en un directorio, opcionalmente filtrando por tipo."""
    if not directory_path.exists():
        return 0, f"Carpeta no encontrada: {directory_path}"

    try:
        count = 0
        for entry in directory_path.iterdir():  # iterdir() no usa patrones — maneja espacios en nombres
            if not entry.is_file():
                continue
            if file_type == "image":
                if entry.suffix.lower() in IMG_EXTS:
                    count += 1
            elif file_type == "label":
                if entry.suffix.lower() == ".txt":
                    count += 1
            else:
                count += 1
        return count, None

    except OSError:
        time.sleep(2)
        try:
            count = 0
            for entry in directory_path.iterdir():
                if not entry.is_file():
                    continue
                if file_type == "image":
                    if entry.suffix.lower() in IMG_EXTS:
                        count += 1
                elif file_type == "label":
                    if entry.suffix.lower() == ".txt":
                        count += 1
                else:
                    count += 1
            return count, None
        except OSError:
            return 0, f"Error de I/O (Drive desconectado?): {directory_path}"

print("📊 Conteo de archivos en directorios:")
print("=" * 40)

paths = [
    (YOLO / "images" / "train", "image", "imágenes"),
    (YOLO / "images" / "val",   "image", "imágenes"),
    (YOLO / "labels" / "train", "label", "labels (.txt)"),
    (YOLO / "labels" / "val",   "label", "labels (.txt)"),
]

for path, ftype, label in paths:
    count, error = count_files_in_dir(path, file_type=ftype)
    if error:
        print(f"  ❌ {error}")
    else:
        print(f"  📁 {path.relative_to(BASE)}: {count:,} {label}")

print("=" * 40)

📊 Conteo de archivos en directorios:
  📁 YOLO_DATASET_UNIFIED/images/train: 87,130 imágenes
  📁 YOLO_DATASET_UNIFIED/images/val: 30,268 imágenes
  📁 YOLO_DATASET_UNIFIED/labels/train: 87,128 labels (.txt)
  📁 YOLO_DATASET_UNIFIED/labels/val: 30,268 labels (.txt)


In [4]:
# ============================================================
# Diagnóstico: estructura real de YOLO_DATASET_UNIFIED
# ============================================================
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
YOLO = BASE / "YOLO_DATASET_UNIFIED"

print(f"¿Existe BASE?  {BASE.exists()}  → {BASE}")
print(f"¿Existe YOLO?  {YOLO.exists()}  → {YOLO}")
print()

if YOLO.exists():
    print("📂 Contenido de YOLO_DATASET_UNIFIED (2 niveles):")
    for p in sorted(YOLO.rglob("*")):
        depth = len(p.relative_to(YOLO).parts)
        if depth <= 2:
            indent = "  " * depth
            marker = "📁" if p.is_dir() else "📄"
            print(f"{indent}{marker} {p.name}")
else:
    print("❌ YOLO_DATASET_UNIFIED no existe. Listando contenido de BASE:")
    if BASE.exists():
        for p in sorted(BASE.iterdir()):
            print(f"  {'📁' if p.is_dir() else '📄'} {p.name}")
    else:
        print("❌ BASE tampoco existe. Verifica que Drive esté montado.")
        print("\n💡 Ejecuta primero:")
        print("   from google.colab import drive")
        print("   drive.mount('/content/drive')")

¿Existe BASE?  True  → /content/drive/MyDrive/TFM/CAIDAS
¿Existe YOLO?  True  → /content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED

📂 Contenido de YOLO_DATASET_UNIFIED (2 niveles):
  📄 ._images
  📄 ._labels
  📄 data.yaml
  📄 data_backup_20260606_165209.yaml
  📁 images
    📄 ._train
    📄 ._val
    📁 train
    📁 val
  📁 labels
    📄 ._train
    📄 ._val
    📁 train
    📁 val


In [5]:
# ============================================================
# Diagnóstico profundo: buscar imágenes y labels en TFM/CAIDAS
# ============================================================
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("📂 Estructura completa de TFM/CAIDAS (hasta 4 niveles):")
print("=" * 60)

for p in sorted(BASE.rglob("*")):
    depth = len(p.relative_to(BASE).parts)
    if depth > 4:
        continue
    indent = "  " * depth
    if p.is_dir():
        # Contar archivos dentro para dar contexto
        try:
            n_files = sum(1 for f in p.iterdir() if f.is_file())
            print(f"{indent}📁 {p.name}/  ({n_files} archivos)")
        except:
            print(f"{indent}📁 {p.name}/")
    else:
        if depth <= 3:  # Solo mostrar archivos en niveles altos
            print(f"{indent}📄 {p.name}")

print("=" * 60)

📂 Estructura completa de TFM/CAIDAS (hasta 4 niveles):
  📁 COCO_2017/  (1 archivos)
    📄 ._poses
    📁 poses/  (3 archivos)
      📄 ._dataset
      📄 ._filtrar_poses.py
      📁 annotations/  (6 archivos)
      📁 dataset/  (4 archivos)
        📁 train/  (2 archivos)
        📁 val/  (2 archivos)
      📄 filtrar_poses.py
      📁 train2017/  (118289 archivos)
  📁 YOLO_DATASET_UNIFIED/  (4 archivos)
    📄 ._images
    📄 ._labels
    📄 data.yaml
    📄 data_backup_20260606_165209.yaml
    📁 images/  (2 archivos)
      📄 ._train
      📄 ._val
      📁 train/  (87130 archivos)
      📁 val/  (30268 archivos)
    📁 labels/  (2 archivos)
      📄 ._train
      📄 ._val
      📁 train/  (87128 archivos)
      📁 val/  (30268 archivos)


In [7]:
# ============================================================
# Redistribución 70/15/15 — YOLO_DATASET_UNIFIED (CAÍDAS)
# ============================================================
import shutil
import random
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/CAIDAS")
YOLO = BASE / "YOLO_DATASET_UNIFIED"

IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED         = 42

def is_real_file(p: Path, valid_exts: set) -> bool:
    return p.is_file() and not p.name.startswith("._") and p.suffix.lower() in valid_exts

def safe_copy(src: Path, dst: Path):
    """Copia solo si src y dst son archivos distintos."""
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

# ── 1. Recopilar pares completos de train + val actuales ──────
src_splits = ["train", "val"]
all_pairs: dict[str, Path] = {}

for split in src_splits:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split
    if not img_dir.exists():
        print(f"⚠️  No encontrada: {img_dir}"); continue
    for img_path in img_dir.iterdir():
        if not is_real_file(img_path, IMG_EXTS): continue
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists() and not lbl_path.name.startswith("._"):
            all_pairs[img_path.stem] = img_path

print(f"✅ Pares imagen+label válidos: {len(all_pairs):,}")

# ── 2. Mezclar y partir ───────────────────────────────────────
stems = sorted(all_pairs.keys())
random.seed(SEED)
random.shuffle(stems)

n_total = len(stems)
n_train = int(n_total * SPLIT_RATIOS["train"])
n_val   = int(n_total * SPLIT_RATIOS["val"])
n_test  = n_total - n_train - n_val

splits = {
    "train": stems[:n_train],
    "val":   stems[n_train : n_train + n_val],
    "test":  stems[n_train + n_val :],
}

print("\n📐 Distribución planificada:")
for s, lst in splits.items():
    print(f"  {s:5s}: {len(lst):,}  ({len(lst)/n_total*100:.1f}%)")

# ── 3. Crear carpetas destino ─────────────────────────────────
for split in ("train", "val", "test"):
    (YOLO / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO / "labels" / split).mkdir(parents=True, exist_ok=True)

# ── 4. Helper ─────────────────────────────────────────────────
def find_label(stem: str) -> Path | None:
    for sp in src_splits:
        p = YOLO / "labels" / sp / (stem + ".txt")
        if p.exists() and not p.name.startswith("._"):
            return p
    return None

# ── 5. Copiar val y test ──────────────────────────────────────
errors = []

for split in ("val", "test"):
    dst_img = YOLO / "images" / split
    dst_lbl = YOLO / "labels" / split
    for stem in splits[split]:
        img_src = all_pairs[stem]
        lbl_src = find_label(stem)
        if lbl_src is None:
            errors.append(stem); continue
        safe_copy(img_src, dst_img / img_src.name)
        safe_copy(lbl_src, dst_lbl / lbl_src.name)
    print(f"✅ Copiados → {split}")

# ── 6. Mover a train los que venían de val original ───────────
for stem in splits["train"]:
    img_src = all_pairs[stem]
    lbl_src = find_label(stem)
    if img_src.parent == YOLO / "images" / "train":
        continue
    dst_img = YOLO / "images" / "train" / img_src.name
    dst_lbl = YOLO / "labels" / "train" / (stem + ".txt")
    if not dst_img.exists():
        safe_copy(img_src, dst_img)
    if lbl_src and not dst_lbl.exists():
        safe_copy(lbl_src, dst_lbl)

print("✅ Copiados → train (desde val original)")

# ── 7. Limpiar train ──────────────────────────────────────────
non_train = set(splits["val"]) | set(splits["test"])
removed = 0
for subdir, exts in [("images", IMG_EXTS), ("labels", {".txt"})]:
    for p in list((YOLO / subdir / "train").iterdir()):
        if is_real_file(p, exts) and p.stem in non_train:
            p.unlink(); removed += 1
print(f"🧹 Eliminados de train (reasignados): {removed:,}")

# ── 8. Limpiar val original ───────────────────────────────────
val_set = set(splits["val"])
for subdir, exts in [("images", IMG_EXTS), ("labels", {".txt"})]:
    for p in list((YOLO / subdir / "val").iterdir()):
        if is_real_file(p, exts) and p.stem not in val_set:
            p.unlink()
print("🧹 val original limpio")

# ── 9. Resumen final ──────────────────────────────────────────
print("\n📊 Verificación final:")
print("=" * 50)
for split in ("train", "val", "test"):
    n_img = sum(1 for p in (YOLO / "images" / split).iterdir()
                if is_real_file(p, IMG_EXTS))
    n_lbl = sum(1 for p in (YOLO / "labels" / split).iterdir()
                if is_real_file(p, {".txt"}))
    status = "✅" if n_img == n_lbl else "⚠️  DESBALANCE"
    print(f"  {status} {split:5s} → {n_img:,} imágenes | {n_lbl:,} labels")

if errors:
    print(f"\n⚠️  {len(errors)} stems sin par: {errors[:5]}")
else:
    print("\n✅ Sin errores de paridad.")
print("=" * 50)

✅ Pares imagen+label válidos: 53,038

📐 Distribución planificada:
  train: 37,126  (70.0%)
  val  : 7,955  (15.0%)
  test : 7,957  (15.0%)
✅ Copiados → val
✅ Copiados → test
✅ Copiados → train (desde val original)
🧹 Eliminados de train (reasignados): 29,116
🧹 val original limpio

📊 Verificación final:
  ⚠️  DESBALANCE train → 37,157 imágenes | 37,126 labels
  ✅ val   → 7,955 imágenes | 7,955 labels
  ✅ test  → 7,957 imágenes | 7,957 labels

✅ Sin errores de paridad.


In [7]:
# --- Celda 8: Conteo de archivos por subcarpeta ---
import subprocess
from pathlib import Path

# Montar Drive si no está montado
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")

print("📊 Conteo de archivos por subcarpeta")
print("=" * 50)

total_img = 0
total_lbl = 0

for split in ["train", "val", "test"]:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split

    # Usar ls -1 con wc -l para contar (menos propenso a timeout que find)
    try:
        r_img = subprocess.run(f"ls -1 '{img_dir}' 2>/dev/null | grep -cEi '\.(jpg|jpeg|png|bmp)$' || echo 0",
            shell=True, capture_output=True, text=True, timeout=120)
        n_img = int(r_img.stdout.strip()) if img_dir.exists() else 0
    except Exception:
        n_img = 0

    try:
        r_lbl = subprocess.run(f"ls -1 '{lbl_dir}' 2>/dev/null | grep -c '\.txt$' || echo 0",
            shell=True, capture_output=True, text=True, timeout=120)
        n_lbl = int(r_lbl.stdout.strip()) if lbl_dir.exists() else 0
    except Exception:
        n_lbl = 0

    status = "✅" if n_img == n_lbl else "⚠️ DESBALANCE"
    print(f"  {status} {split:5s} → {n_img:,} imágenes | {n_lbl:,} etiquetas")
    total_img += n_img
    total_lbl += n_lbl

print("=" * 50)
print(f"  TOTAL  → {total_img:,} imágenes | {total_lbl:,} etiquetas")

<>:24: SyntaxWarning: invalid escape sequence '\.'
<>:31: SyntaxWarning: invalid escape sequence '\.'
<>:24: SyntaxWarning: invalid escape sequence '\.'
<>:31: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_2427/1090243080.py:24: SyntaxWarning: invalid escape sequence '\.'
  r_img = subprocess.run(f"ls -1 '{img_dir}' 2>/dev/null | grep -cEi '\.(jpg|jpeg|png|bmp)$' || echo 0",
/tmp/ipykernel_2427/1090243080.py:31: SyntaxWarning: invalid escape sequence '\.'
  r_lbl = subprocess.run(f"ls -1 '{lbl_dir}' 2>/dev/null | grep -c '\.txt$' || echo 0",


📊 Conteo de archivos por subcarpeta
  ⚠️ DESBALANCE train → 37,157 imágenes | 37,126 etiquetas
  ⚠️ DESBALANCE val   → 8,261 imágenes | 8,294 etiquetas
  ✅ test  → 7,957 imágenes | 7,957 etiquetas
  TOTAL  → 53,375 imágenes | 53,377 etiquetas


In [8]:
# ==== Celda 9: Corrección de desbalance ====
# Elimina imágenes y etiquetas huérfanas (sin contrapartida) en cada split
# Usa subprocess para evitar timeout de Drive con >37K archivos
import subprocess
import os
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")
IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp"]

def list_stems_subprocess(directory, ext_filter):
    """Lista stems de archivos usando subprocess para evitar timeout en Drive."""
    if not directory.exists():
        return set()
    # find devuelve rutas completas; extraemos el stem
    ext_args = " -o ".join([f'-name "*{e}"' for e in ext_filter])
    cmd = f'find "{directory}" -maxdepth 1 \( {ext_args} \) -printf "%f\n"'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    stems = set()
    for fname in r.stdout.strip().splitlines():
        stem = Path(fname).stem
        if stem:
            stems.add(stem)
    return stems

print("🔧 Corrección de desbalance — identificando archivos huérfanos")
print("=" * 60)
print("  (Esto puede tardar ~2-5 min en leer 37K+ archivos de Drive)")
print()

total_removed = 0

for split in ["train", "val", "test"]:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split

    if not img_dir.exists() or not lbl_dir.exists():
        print(f"  [{split}] Directorio no encontrado, omitiendo.")
        continue

    print(f"  Leyendo stems de [{split}] images ...", end=" ", flush=True)
    img_stems = list_stems_subprocess(img_dir, IMG_EXTS)
    print(f"{len(img_stems):,}")

    print(f"  Leyendo stems de [{split}] labels ...", end=" ", flush=True)
    lbl_stems = list_stems_subprocess(lbl_dir, [".txt"])
    print(f"{len(lbl_stems):,}")

    orphan_imgs = img_stems - lbl_stems   # imágenes sin etiqueta
    orphan_lbls = lbl_stems - img_stems   # etiquetas sin imagen

    removed = 0

    # Eliminar imágenes huérfanas
    for stem in orphan_imgs:
        for ext in IMG_EXTS:
            candidate = img_dir / (stem + ext)
            if candidate.exists():
                candidate.unlink()
                removed += 1
                break

    # Eliminar etiquetas huérfanas
    for stem in orphan_lbls:
        candidate = lbl_dir / (stem + ".txt")
        if candidate.exists():
            candidate.unlink()
            removed += 1

    status = "✅" if removed == 0 else "🗑️ "
    print(f"  {status} [{split:5s}] imágenes huérfanas: {len(orphan_imgs):3d} | etiquetas huérfanas: {len(orphan_lbls):3d} | eliminados: {removed}")
    print()
    total_removed += removed

print("=" * 60)
print(f"  Total archivos eliminados: {total_removed}")
print()

# Verificación final con subprocess (rápido)
print("📊 Conteo final post-corrección")
print("=" * 60)
for split in ["train", "val", "test"]:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split
    r_img = subprocess.run(
        f'find "{img_dir}" -maxdepth 1 -name "*.jpg" -o -name "*.jpeg" -o -name "*.png" -o -name "*.bmp" | wc -l',
        shell=True, capture_output=True, text=True, timeout=120)
    r_lbl = subprocess.run(
        f'find "{lbl_dir}" -maxdepth 1 -name "*.txt" | wc -l',
        shell=True, capture_output=True, text=True, timeout=120)
    n_img = int(r_img.stdout.strip()) if img_dir.exists() else 0
    n_lbl = int(r_lbl.stdout.strip()) if lbl_dir.exists() else 0
    ok = "✅" if n_img == n_lbl else "⚠️  DESBALANCE"
    print(f"  {ok} {split:5s} → {n_img:,} imágenes | {n_lbl:,} etiquetas")
print("=" * 60)

<>:21: SyntaxWarning: invalid escape sequence '\('
<>:21: SyntaxWarning: invalid escape sequence '\)'
<>:21: SyntaxWarning: invalid escape sequence '\('
<>:21: SyntaxWarning: invalid escape sequence '\)'
/tmp/ipykernel_2427/877543218.py:21: SyntaxWarning: invalid escape sequence '\('
  cmd = f'find "{directory}" -maxdepth 1 \( {ext_args} \) -printf "%f\n"'
/tmp/ipykernel_2427/877543218.py:21: SyntaxWarning: invalid escape sequence '\)'
  cmd = f'find "{directory}" -maxdepth 1 \( {ext_args} \) -printf "%f\n"'


🔧 Corrección de desbalance — identificando archivos huérfanos
  (Esto puede tardar ~2-5 min en leer 37K+ archivos de Drive)

  Leyendo stems de [train] images ... 75,895
  Leyendo stems de [train] labels ... 75,893
  🗑️  [train] imágenes huérfanas:   2 | etiquetas huérfanas:   0 | eliminados: 2

  Leyendo stems de [val] images ... 22,195
  Leyendo stems de [val] labels ... 22,228
  🗑️  [val  ] imágenes huérfanas:  96 | etiquetas huérfanas: 129 | eliminados: 225

  Leyendo stems de [test] images ... 7,957
  Leyendo stems de [test] labels ... 7,957
  ✅ [test ] imágenes huérfanas:   0 | etiquetas huérfanas:   0 | eliminados: 0

  Total archivos eliminados: 227

📊 Conteo final post-corrección
  ⚠️  DESBALANCE train → 75,922 imágenes | 75,893 etiquetas
  ✅ val   → 22,099 imágenes | 22,099 etiquetas
  ✅ test  → 7,957 imágenes | 7,957 etiquetas


In [9]:
# ==== Celda 10: Diagnóstico y corrección final del desbalance en train ====
# Usa 'comm' para comparación precisa de stems (sort + diff)
import subprocess
from pathlib import Path

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")
img_dir = YOLO / "images" / "train"
lbl_dir = YOLO / "labels" / "train"

print("🔍 Diagnóstico preciso del desbalance en train")
print("=" * 55)
print("  Obteniendo lista de stems con find -name '*.jpg' ...")

# Solo *.jpg — la extensión real del dataset (sin -o para evitar bugs)
# Guarda los stems ordenados en archivos temporales para usar comm
r1 = subprocess.run(
    f'find "{img_dir}" -maxdepth 1 -name "*.jpg" -printf "%f\n" | sed "s/\.jpg$//" | sort > /tmp/img_stems.txt',
    shell=True, capture_output=True, text=True, timeout=300)
r2 = subprocess.run(
    f'find "{lbl_dir}" -maxdepth 1 -name "*.txt" -printf "%f\n" | sed "s/\.txt$//" | sort > /tmp/lbl_stems.txt',
    shell=True, capture_output=True, text=True, timeout=300)

# Contar stems
n_img = int(subprocess.run('wc -l < /tmp/img_stems.txt', shell=True, capture_output=True, text=True).stdout.strip())
n_lbl = int(subprocess.run('wc -l < /tmp/lbl_stems.txt', shell=True, capture_output=True, text=True).stdout.strip())
print(f"  Stems únicos → images: {n_img:,} | labels: {n_lbl:,}")

# comm -23: líneas solo en img (huérfanas de imagen)
# comm -13: líneas solo en lbl (huérfanas de etiqueta)
r_orphan_img = subprocess.run(
    'comm -23 /tmp/img_stems.txt /tmp/lbl_stems.txt',
    shell=True, capture_output=True, text=True, timeout=60)
r_orphan_lbl = subprocess.run(
    'comm -13 /tmp/img_stems.txt /tmp/lbl_stems.txt',
    shell=True, capture_output=True, text=True, timeout=60)

orphan_imgs = [s for s in r_orphan_img.stdout.strip().splitlines() if s]
orphan_lbls = [s for s in r_orphan_lbl.stdout.strip().splitlines() if s]
print(f"  Imágenes sin etiqueta: {len(orphan_imgs)}")
print(f"  Etiquetas sin imagen:  {len(orphan_lbls)}")

if len(orphan_imgs) > 0:
    print(f"  Primeras 5 imágenes huérfanas: {orphan_imgs[:5]}")
if len(orphan_lbls) > 0:
    print(f"  Primeras 5 etiquetas huérfanas: {orphan_lbls[:5]}")

# Eliminar orphans
removed = 0
for stem in orphan_imgs:
    p = img_dir / (stem + ".jpg")
    if p.exists():
        p.unlink()
        removed += 1

for stem in orphan_lbls:
    p = lbl_dir / (stem + ".txt")
    if p.exists():
        p.unlink()
        removed += 1

print(f"\n  🗑️  Eliminados: {removed} archivo(s)")

# Verificación final confiable
n_img_final = int(subprocess.run(
    f'find "{img_dir}" -maxdepth 1 -name "*.jpg" | wc -l',
    shell=True, capture_output=True, text=True, timeout=120).stdout.strip())
n_lbl_final = int(subprocess.run(
    f'find "{lbl_dir}" -maxdepth 1 -name "*.txt" | wc -l',
    shell=True, capture_output=True, text=True, timeout=120).stdout.strip())

status = "✅" if n_img_final == n_lbl_final else "⚠️  DESBALANCE"
print(f"\n  {status} train → {n_img_final:,} imágenes | {n_lbl_final:,} etiquetas")
print("=" * 55)

<>:17: SyntaxWarning: invalid escape sequence '\.'
<>:20: SyntaxWarning: invalid escape sequence '\.'
<>:17: SyntaxWarning: invalid escape sequence '\.'
<>:20: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_2427/989465204.py:17: SyntaxWarning: invalid escape sequence '\.'
  f'find "{img_dir}" -maxdepth 1 -name "*.jpg" -printf "%f\n" | sed "s/\.jpg$//" | sort > /tmp/img_stems.txt',
/tmp/ipykernel_2427/989465204.py:20: SyntaxWarning: invalid escape sequence '\.'
  f'find "{lbl_dir}" -maxdepth 1 -name "*.txt" -printf "%f\n" | sed "s/\.txt$//" | sort > /tmp/lbl_stems.txt',


🔍 Diagnóstico preciso del desbalance en train
  Obteniendo lista de stems con find -name '*.jpg' ...
  Stems únicos → images: 75,893 | labels: 75,893
  Imágenes sin etiqueta: 0
  Etiquetas sin imagen:  0

  🗑️  Eliminados: 0 archivo(s)

  ✅ train → 75,893 imágenes | 75,893 etiquetas


In [10]:
# ==== Celda 10: Conteo final de verificación ====
import subprocess
from pathlib import Path

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")

print("📊 Conteo final verificado (find con una sola extensión)")
print("=" * 60)
total_img = 0
total_lbl = 0
for split in ["train", "val", "test"]:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split
    # find con UNA extensión en comillas simples — evita doble conteo
    r_img = subprocess.run(
        ["find", str(img_dir), "-maxdepth", "1", "-name", "*.jpg"],
        capture_output=True, text=True, timeout=120)
    r_lbl = subprocess.run(
        ["find", str(lbl_dir), "-maxdepth", "1", "-name", "*.txt"],
        capture_output=True, text=True, timeout=120)
    n_img = len([l for l in r_img.stdout.strip().splitlines() if l])
    n_lbl = len([l for l in r_lbl.stdout.strip().splitlines() if l])
    ok = "✅" if n_img == n_lbl else "⚠️  DESBALANCE"
    print(f"  {ok} {split:5s} → {n_img:,} imágenes | {n_lbl:,} etiquetas")
    total_img += n_img
    total_lbl += n_lbl

print("=" * 60)
print(f"  TOTAL   → {total_img:,} imágenes | {total_lbl:,} etiquetas")
ok_total = "✅ Dataset completamente balanceado" if total_img == total_lbl else "⚠️ Desbalance total"
print(f"  {ok_total}")
print("=" * 60)

📊 Conteo final verificado (find con una sola extensión)
  ✅ train → 75,893 imágenes | 75,893 etiquetas
  ⚠️  DESBALANCE val   → 22,025 imágenes | 22,099 etiquetas
  ⚠️  DESBALANCE test  → 7,948 imágenes | 7,957 etiquetas
  TOTAL   → 105,866 imágenes | 105,949 etiquetas
  ⚠️ Desbalance total


In [2]:
# ==== Corrección segura de desbalance en val y test ====
from pathlib import Path
import shutil

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")
QUARANTINE = YOLO / "_orphan_files_quarantine"
QUARANTINE.mkdir(exist_ok=True)

splits = ["train", "val", "test"]

print("🔍 Diagnóstico y corrección segura de huérfanos")
print("=" * 70)

total_moved = 0

for split in splits:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split

    q_img_dir = QUARANTINE / "images" / split
    q_lbl_dir = QUARANTINE / "labels" / split
    q_img_dir.mkdir(parents=True, exist_ok=True)
    q_lbl_dir.mkdir(parents=True, exist_ok=True)

    img_files = list(img_dir.glob("*.jpg"))
    lbl_files = list(lbl_dir.glob("*.txt"))

    img_stems = {p.stem for p in img_files}
    lbl_stems = {p.stem for p in lbl_files}

    imgs_without_label = sorted(img_stems - lbl_stems)
    labels_without_img = sorted(lbl_stems - img_stems)

    print(f"\n📁 Split: {split}")
    print(f"  Imágenes: {len(img_files):,}")
    print(f"  Labels:   {len(lbl_files):,}")
    print(f"  Imágenes sin label: {len(imgs_without_label)}")
    print(f"  Labels sin imagen:  {len(labels_without_img)}")

    if imgs_without_label[:5]:
        print(f"  Ejemplos imágenes sin label: {imgs_without_label[:5]}")

    if labels_without_img[:5]:
        print(f"  Ejemplos labels sin imagen: {labels_without_img[:5]}")

    # Mover imágenes huérfanas
    for stem in imgs_without_label:
        src = img_dir / f"{stem}.jpg"
        dst = q_img_dir / f"{stem}.jpg"
        if src.exists():
            shutil.move(str(src), str(dst))
            total_moved += 1

    # Mover labels huérfanos
    for stem in labels_without_img:
        src = lbl_dir / f"{stem}.txt"
        dst = q_lbl_dir / f"{stem}.txt"
        if src.exists():
            shutil.move(str(src), str(dst))
            total_moved += 1

print("\n" + "=" * 70)
print(f"✅ Archivos huérfanos movidos a cuarentena: {total_moved}")
print(f"📦 Carpeta de cuarentena: {QUARANTINE}")
print("=" * 70)

🔍 Diagnóstico y corrección segura de huérfanos

📁 Split: train
  Imágenes: 0
  Labels:   0
  Imágenes sin label: 0
  Labels sin imagen:  0

📁 Split: val
  Imágenes: 22,025
  Labels:   22,099
  Imágenes sin label: 0
  Labels sin imagen:  74
  Ejemplos labels sin imagen: ['._FALL_DETECTION_not fallen020', '._FALL_DETECTION_not fallen021', '._FALL_DETECTION_not fallen022', '._FALL_DETECTION_not fallen023', '._FALL_DETECTION_not fallen024']

📁 Split: test
  Imágenes: 7,948
  Labels:   7,957
  Imágenes sin label: 0
  Labels sin imagen:  9
  Ejemplos labels sin imagen: ['FALL_DETECTION_not fallen020', 'FALL_DETECTION_not fallen024', 'FALL_DETECTION_not fallen025', 'KUMAR_FALL_DETECTION_not fallen021', 'KUMAR_FALL_DETECTION_not fallen025']

✅ Archivos huérfanos movidos a cuarentena: 83
📦 Carpeta de cuarentena: /content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED/_orphan_files_quarantine


In [3]:
# ==== Verificación final después de la corrección ====
from pathlib import Path

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")

print("📊 Verificación final image-label")
print("=" * 60)

total_img = 0
total_lbl = 0

for split in ["train", "val", "test"]:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split

    n_img = len(list(img_dir.glob("*.jpg")))
    n_lbl = len(list(lbl_dir.glob("*.txt")))

    ok = "✅" if n_img == n_lbl else "⚠️ DESBALANCE"

    print(f"  {ok} {split:5s} → {n_img:,} imágenes | {n_lbl:,} etiquetas")

    total_img += n_img
    total_lbl += n_lbl

print("=" * 60)
print(f"  TOTAL   → {total_img:,} imágenes | {total_lbl:,} etiquetas")
print("  ✅ Dataset balanceado" if total_img == total_lbl else "  ⚠️ Todavía hay desbalance")
print("=" * 60)

📊 Verificación final image-label
  ✅ train → 75,893 imágenes | 75,893 etiquetas
  ✅ val   → 22,025 imágenes | 22,025 etiquetas
  ✅ test  → 7,948 imágenes | 7,948 etiquetas
  TOTAL   → 105,866 imágenes | 105,866 etiquetas
  ✅ Dataset balanceado


In [5]:
# ==== Limpieza de archivos ocultos/metadatos macOS en dataset YOLO ====
from pathlib import Path
import shutil
from datetime import datetime

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")
QUARANTINE = YOLO / f"_metadata_quarantine_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
QUARANTINE.mkdir(exist_ok=True)

splits = ["train", "val", "test"]
folders = ["images", "labels"]

moved = 0

print("🧹 Buscando archivos ocultos/metadatos tipo ._ y .DS_Store")
print("=" * 70)

for folder in folders:
    for split in splits:
        src_dir = YOLO / folder / split
        dst_dir = QUARANTINE / folder / split
        dst_dir.mkdir(parents=True, exist_ok=True)

        for p in src_dir.iterdir():
            if p.name.startswith("._") or p.name == ".DS_Store" or p.name.startswith("."):
                shutil.move(str(p), str(dst_dir / p.name))
                moved += 1
                print(f"  Movido: {folder}/{split}/{p.name}")

print("=" * 70)
print(f"✅ Archivos ocultos/metadatos movidos: {moved}")
print(f"📦 Cuarentena: {QUARANTINE}")
print("=" * 70)

Streaming output truncated to the last 5000 lines.
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0182.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (67)_f0332.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0037.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0040.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0353.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0099.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0111.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0215.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0256.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0173.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0002.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0047.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0325.txt
  Movido: labels/val/._IMVIA_Coffee_room_02_video (68)_f0127.txt
  Movido: labels/val/._IMVIA_Coffee_roo

In [8]:
# ==== Redistribución YOLO robusta a 70/15/15 con renombrado ante colisiones ====
from pathlib import Path
import shutil
import random
import json
from datetime import datetime

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")

IMG_EXT = ".jpg"
LBL_EXT = ".txt"
SPLITS = ["train", "val", "test"]
SEED = 42

random.seed(SEED)

images_root = YOLO / "images"
labels_root = YOLO / "labels"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
quarantine = YOLO / f"_quarantine_orphans_{timestamp}"
log_path = YOLO / f"redistribution_70_15_15_safe_log_{timestamp}.json"


def is_valid_file(p, ext):
    return (
        p.is_file()
        and p.suffix.lower() == ext
        and not p.name.startswith(".")
        and not p.name.startswith("._")
    )


def get_valid_files(folder, ext):
    return [p for p in folder.iterdir() if is_valid_file(p, ext)]


def quarantine_orphans():
    print("🧹 Revisando huérfanos antes de redistribuir...")
    moved = 0

    for split in SPLITS:
        img_dir = images_root / split
        lbl_dir = labels_root / split

        imgs = get_valid_files(img_dir, IMG_EXT)
        lbls = get_valid_files(lbl_dir, LBL_EXT)

        img_stems = {p.stem for p in imgs}
        lbl_stems = {p.stem for p in lbls}

        orphan_imgs = sorted(img_stems - lbl_stems)
        orphan_lbls = sorted(lbl_stems - img_stems)

        q_img_dir = quarantine / "images" / split
        q_lbl_dir = quarantine / "labels" / split
        q_img_dir.mkdir(parents=True, exist_ok=True)
        q_lbl_dir.mkdir(parents=True, exist_ok=True)

        for stem in orphan_imgs:
            src = img_dir / f"{stem}{IMG_EXT}"
            if src.exists():
                shutil.move(str(src), str(q_img_dir / src.name))
                moved += 1

        for stem in orphan_lbls:
            src = lbl_dir / f"{stem}{LBL_EXT}"
            if src.exists():
                shutil.move(str(src), str(q_lbl_dir / src.name))
                moved += 1

        print(
            f"  {split:5s}: imágenes sin label={len(orphan_imgs):,}, "
            f"labels sin imagen={len(orphan_lbls):,}"
        )

    print(f"✅ Huérfanos movidos a cuarentena: {moved:,}")
    if moved:
        print(f"📦 Cuarentena: {quarantine}")


def get_pairs(split):
    img_dir = images_root / split
    lbl_dir = labels_root / split

    imgs = get_valid_files(img_dir, IMG_EXT)
    lbls = get_valid_files(lbl_dir, LBL_EXT)

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    if img_stems != lbl_stems:
        raise ValueError(f"Todavía hay desbalance interno en {split}")

    return sorted(img_stems)


def unique_destination_stem(stem, dst_split, src_split):
    """
    Si el stem ya existe en destino, crea un nombre único.
    Ejemplo:
    KUMAR_FALL_DETECTION_fall005__from_val__001
    """
    candidate = stem
    i = 1

    while True:
        dst_img = images_root / dst_split / f"{candidate}{IMG_EXT}"
        dst_lbl = labels_root / dst_split / f"{candidate}{LBL_EXT}"

        if not dst_img.exists() and not dst_lbl.exists():
            return candidate

        candidate = f"{stem}__from_{src_split}__{i:03d}"
        i += 1


def move_pair_safe(stem, src_split, dst_split):
    src_img = images_root / src_split / f"{stem}{IMG_EXT}"
    src_lbl = labels_root / src_split / f"{stem}{LBL_EXT}"

    if not src_img.exists() or not src_lbl.exists():
        raise FileNotFoundError(f"Falta imagen o label para {stem} en {src_split}")

    final_stem = unique_destination_stem(stem, dst_split, src_split)

    dst_img = images_root / dst_split / f"{final_stem}{IMG_EXT}"
    dst_lbl = labels_root / dst_split / f"{final_stem}{LBL_EXT}"

    shutil.move(str(src_img), str(dst_img))
    shutil.move(str(src_lbl), str(dst_lbl))

    return final_stem


# 1) Limpiar huérfanos
quarantine_orphans()

# 2) Recalcular estado actual
pairs_by_split = {split: get_pairs(split) for split in SPLITS}
counts = {split: len(pairs_by_split[split]) for split in SPLITS}
total = sum(counts.values())

targets = {
    "train": round(total * 0.70),
    "val": round(total * 0.15),
}
targets["test"] = total - targets["train"] - targets["val"]

print("\n📊 Distribución actual")
print("=" * 70)
for split in SPLITS:
    print(f"  {split:5s}: {counts[split]:,} pares ({counts[split] / total * 100:.2f}%)")

print("\n🎯 Distribución objetivo")
print("=" * 70)
for split in SPLITS:
    print(f"  {split:5s}: {targets[split]:,} pares ({targets[split] / total * 100:.2f}%)")

diffs = {split: counts[split] - targets[split] for split in SPLITS}

excess = {s: diffs[s] for s in SPLITS if diffs[s] > 0}
deficit = {s: -diffs[s] for s in SPLITS if diffs[s] < 0}

moves = []

for src_split, extra in excess.items():
    stems = pairs_by_split[src_split][:]
    random.shuffle(stems)
    cursor = 0

    for dst_split in list(deficit.keys()):
        needed = deficit[dst_split]

        if needed <= 0 or extra <= 0:
            continue

        n = min(extra, needed)
        selected = stems[cursor:cursor + n]
        cursor += n

        for stem in selected:
            moves.append({
                "stem": stem,
                "from": src_split,
                "to": dst_split,
            })

        extra -= n
        deficit[dst_split] -= n

print("\n🚚 Movimientos planificados")
print("=" * 70)
summary = {}
for m in moves:
    key = f"{m['from']} -> {m['to']}"
    summary[key] = summary.get(key, 0) + 1

if summary:
    for key, n in summary.items():
        print(f"  {key}: {n:,} pares")
else:
    print("  No hay movimientos pendientes.")

confirmar = input("\nEscribe SI para ejecutar los movimientos: ").strip().upper()

if confirmar != "SI":
    print("❌ Operación cancelada. No se movió ningún archivo.")
else:
    executed = []
    renamed = []

    print("\n🚚 Moviendo pares...")
    for i, m in enumerate(moves, start=1):
        new_stem = move_pair_safe(m["stem"], m["from"], m["to"])

        record = {
            **m,
            "new_stem": new_stem,
        }
        executed.append(record)

        if new_stem != m["stem"]:
            renamed.append(record)

        if i % 1000 == 0:
            print(f"  Movidos: {i:,}/{len(moves):,}")

    with open(log_path, "w", encoding="utf-8") as f:
        json.dump({
            "timestamp": datetime.now().isoformat(),
            "seed": SEED,
            "total_pairs": total,
            "initial_counts": counts,
            "target_counts": targets,
            "moves_count": len(executed),
            "renamed_count": len(renamed),
            "moves": executed,
        }, f, ensure_ascii=False, indent=2)

    print("\n✅ Redistribución completada")
    print(f"🔁 Pares renombrados por colisión: {len(renamed):,}")
    print(f"🧾 Log guardado en: {log_path}")

🧹 Revisando huérfanos antes de redistribuir...
  train: imágenes sin label=0, labels sin imagen=0
  val  : imágenes sin label=0, labels sin imagen=0
  test : imágenes sin label=0, labels sin imagen=0
✅ Huérfanos movidos a cuarentena: 0

📊 Distribución actual
  train: 36,269 pares (68.16%)
  val  : 8,114 pares (15.25%)
  test : 8,826 pares (16.59%)

🎯 Distribución objetivo
  train: 37,246 pares (70.00%)
  val  : 7,981 pares (15.00%)
  test : 7,982 pares (15.00%)

🚚 Movimientos planificados
  val -> train: 133 pares
  test -> train: 844 pares

Escribe SI para ejecutar los movimientos: SI

🚚 Moviendo pares...

✅ Redistribución completada
🔁 Pares renombrados por colisión: 1
🧾 Log guardado en: /content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED/redistribution_70_15_15_safe_log_20260607_225617.json


In [9]:
# ==== Verificación final YOLO 70/15/15 ====
from pathlib import Path

YOLO = Path("/content/drive/MyDrive/TFM/CAIDAS/YOLO_DATASET_UNIFIED")

SPLITS = ["train", "val", "test"]

def valid_files(folder, suffix):
    return [
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() == suffix
        and not p.name.startswith(".")
        and not p.name.startswith("._")
    ]

counts = {}

print("📊 Verificación final image-label y distribución")
print("=" * 70)

for split in SPLITS:
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split

    imgs = valid_files(img_dir, ".jpg")
    lbls = valid_files(lbl_dir, ".txt")

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    ok = "✅" if img_stems == lbl_stems else "⚠️ DESBALANCE"

    counts[split] = len(img_stems)

    print(f"  {ok} {split:5s} → {len(img_stems):,} imágenes | {len(lbl_stems):,} etiquetas")

total = sum(counts.values())

print("=" * 70)
print(f"  TOTAL → {total:,} pares")

print("\n📐 Distribución")
print("=" * 70)
for split in SPLITS:
    print(f"  {split:5s}: {counts[split]:,} pares → {counts[split] / total * 100:.2f}%")

print("=" * 70)

ok_balance = all(
    len({
        p.stem for p in valid_files(YOLO / "images" / split, ".jpg")
    }) ==
    len({
        p.stem for p in valid_files(YOLO / "labels" / split, ".txt")
    })
    for split in SPLITS
)

if ok_balance:
    print("✅ Dataset listo para YOLOv8")
else:
    print("⚠️ Todavía hay algo que revisar")

📊 Verificación final image-label y distribución
  ✅ train → 37,246 imágenes | 37,246 etiquetas
  ✅ val   → 7,981 imágenes | 7,981 etiquetas
  ✅ test  → 7,982 imágenes | 7,982 etiquetas
  TOTAL → 53,209 pares

📐 Distribución
  train: 37,246 pares → 70.00%
  val  : 7,981 pares → 15.00%
  test : 7,982 pares → 15.00%
✅ Dataset listo para YOLOv8
